In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
import pickle

In [2]:
#Load data set
df = pd.read_csv('data/Churn_Modelling.csv')
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## PREPROCESSING

In [3]:
df = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

In [4]:
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})

In [5]:
geo_encoder = OneHotEncoder(drop='first', sparse_output=False)
geo_encoded = geo_encoder.fit_transform(df[['Geography']])
geo_encoded_df = pd.DataFrame(geo_encoded, columns=geo_encoder.get_feature_names_out(['Geography']), index=df.index)

df = pd.concat([df.drop('Geography', axis=1), geo_encoded_df], axis=1)

with open('pickle/onehot_encoder_geo_salary.pkl', 'wb') as f:
    pickle.dump(geo_encoder, f)

df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,1.0


In [6]:
#save the preprocessed data to a pickle file
with open('pickle/preprocessed_data_salary.pkl', 'wb') as f:
    pickle.dump(df, f)

In [7]:
#EstimatedSalary is the dependent (target) feature, everything else is independent
X = df.drop('EstimatedSalary', axis=1)
y = df['EstimatedSalary']

In [8]:
#train test split
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
#scale the features
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

with open('pickle/scaler_salary.pkl', 'wb') as f:
    pickle.dump(scaler, f)

In [10]:
#saving the train and test data to pickle files
with open('pickle/x_train_salary.pkl', 'wb') as f:
    pickle.dump(x_train, f)

with open('pickle/x_test_salary.pkl', 'wb') as f:
    pickle.dump(x_test, f)

with open('pickle/y_train_salary.pkl', 'wb') as f:
    pickle.dump(y_train, f)

with open('pickle/y_test_salary.pkl', 'wb') as f:
    pickle.dump(y_test, f)

ANN Regression

In [11]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

tf.config.threading.set_intra_op_parallelism_threads(1)
tf.config.threading.set_inter_op_parallelism_threads(1)

In [12]:
model = Sequential([
    tf.keras.Input(shape=(x_train.shape[1],)), #input layer
    Dense(64, activation='relu'), #hidden layer 1
    Dense(32, activation='relu'), #hidden layer 2
    Dense(16, activation='relu'), #hidden layer 3
    Dense(1, activation='linear') #output layer, linear since this is regression
])

2026-07-04 18:44:12.813937: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-07-04 18:44:12.813966: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-07-04 18:44:12.813971: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-07-04 18:44:12.813985: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-07-04 18:44:12.813992: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [13]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,393 (13.25 KB)

 Trainable params: 3,393 (13.25 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
##compile the model
model.compile(optimizer='adam', loss='mean_absolute_error', metrics=['mae'])

In [15]:
#setup the tensorboard
log_dir = "logs/fit_salary/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [16]:
#setup early stopping
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [17]:
#train the model
history = model.fit(x_train, y_train, epochs=100, batch_size=32, validation_split=0.2, callbacks=[early_stopping, tensorboard_callback])

Epoch 1/100


2026-07-04 18:44:13.152579: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 99986.1719 - mae: 99986.1719 - val_loss: 102031.0234 - val_mae: 102031.0234
Epoch 2/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 99265.9922 - mae: 99265.9922 - val_loss: 100190.1016 - val_mae: 100190.1016
Epoch 3/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 95611.6875 - mae: 95611.6875 - val_loss: 94827.2891 - val_mae: 94827.2891
Epoch 4/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 90095.5312 - mae: 90095.5312 - val_loss: 90125.4766 - val_mae: 90125.4766
Epoch 5/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 86468.7266 - mae: 86468.7266 - val_loss: 87422.2891 - val_mae: 87422.2891
Epoch 6/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 83579.1719 - mae: 83579.1719 - val_loss: 84440.6172 - val_mae: 84440.6172
Epoch 7/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 80425.1562 - mae: 80425.1562 - val_loss: 80801.8047 - val_mae: 80801.8047
Epoch 8/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 76

In [18]:
#evaluate on the held out test set
test_loss, test_mae = model.evaluate(x_test, y_test)
print(f"Test MAE: {test_mae:.2f}")

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 50004.7695 - mae: 50004.7695
Test MAE: 50004.77


In [19]:
model.save('models/salary_regression_model.keras')